In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
import scipy

import plotly
from plotly.graph_objects import Scatter
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
init_notebook_mode(connected=False)
import plotly.express as px

import cosmosdr

import cosmosdr.signal_acquisition as s_acq
import cosmosdr.signal_processing as s_proc
from cosmosdr.plotting import create_base_figure

try:
    sdr.close()
except:
    pass

In [ ]:
ADSB_FREQUENCY = 1090e6
# Total number of bits in a single ADSB message
ADSB_BITS = 112
# One microsecond timeslot for one bit, 'on' if signal is within the first half of this slot
ADSB_SLOT_LENGTH = 1 / 1e6

center_freq = ADSB_FREQUENCY
# reccomended upper limit of sample rate. Fast enough to oversample
sample_rate = 2.4e6
n_reads = 32
n_samples = 4096

# choose integer samples per microsecond: 12 samples/us
# This is a good choice, because 6x2 = 12, and 5*2.4=12, so we can sample to 12, then subsample back to 1 block per 0.5us
target_sr = 12e6
samples_per_us = int(round(target_sr/1e6))  # 12


In [ ]:
import numpy as np
from scipy.signal import resample_poly, correlate
from fractions import Fraction

def iq_to_envelope(iq):
    """assumes `iq` is a 1D complex64 numpy array from your SDR, and sr is sample_rate (Hz)"""

    return np.abs(iq)

def resample_to_target(s, orig_sr, target_sr):
    """Try to rebuild the underlying analog waveform that would have produced your samples.
    
    This makes the 'bandlimited reconstruction assumption', which says that, 'I will construct
    the smoothest possible signal I can, on the assumption that there is no received signal above
    the nyquist of the sample rate. 

    Principally, we make an assumption that the frequency is related to the sample rate, which of course may not be
    true, but we know it is in our case.

    Generates the mathematically correct reconstruction of what the signal would look like between samples (assuming
    the original SDR samples met Nyquist(which is 1/2 sample rate) )
    
    Nyquist is the maximum frequency you can capture without ambiguity at a given sample rate..
    """
    
    # find integer up/down using Fraction
    ratio = Fraction(int(target_sr), int(orig_sr)).limit_denominator()
    up, down = ratio.numerator, ratio.denominator
    return resample_poly(s, up, down), target_sr

def get_index_of_highest_peak(s):
    """
    Assumes the input is the result of acquire_signal(), meaning it is an (n, m) ndarray, with n reads

    Returns the index with the highest peak, essentially giving you an index at which there is very likely a signal of some kind.
    """
    index = s.max(axis=1).argmax()
    print("highest index:", index)

    return index
    # s_df = pd.DataFrame(np.abs(s)).T
    
    # index = s_df.max().idxmax()
    
    # # Grab the column with the highest individual peak
    # signal_col = s_df[index]
    # print("highest index:", index)

    # return index


### Sample at target SR, and upsample to higher rate

In [ ]:


# Start up the SDR connection
try:
    sdr.close()
except:
    pass

sdr = s_acq.get_sdr(center_freq=center_freq, sample_rate=orig_sr)

s = s_acq.acquire_signal(sdr, n_reads=n_reads, n_samples=n_samples)

In [ ]:
highest_peak_read = get_index_of_highest_peak(s)
iq = s[highest_peak_read]

In [ ]:

fig = px.bar(pd.DataFrame(iq).abs())

fig.update_traces(marker_line_width = 0,
                  selector=dict(type="bar"))

In [ ]:
# This is the correct way
iq_resampled, _ = resample_to_target(iq, orig_sr, target_sr)
env_resampled = iq_to_envelope(iq_resampled)

In [ ]:
idx = env_resampled.argmax()
n_either_side = 1000


fig = px.bar(pd.DataFrame(env_resampled[idx-n_either_side:idx+n_either_side]).abs())

fig.update_traces(marker_line_width = 0,
                  selector=dict(type="bar"))

### Find the optimal phase starting position
- We don't know the exact timing of the pulses
- It could be anywhere from n, ..., n+11
- We can define the most optimal position as that which has the highest difference between neighboring 6-width blocks

In [ ]:
data = env_resampled[idx-n_either_side:idx+n_either_side]

scores = {}

In [ ]:
max_score = -1
max_score_phase = None

# We check each of the first n starting points, after a while we just get back to the start of the cycle again so we can stop checking
steps_to_check = int(samples_per_us)
scores = {}

# Step through the possible starting points
for phase in range(0, steps_to_check-1):
    data_phase = data[phase:]
    
    # 0,...,0,1,...,1,2,...,2 etc
    indices = np.arange(len(data_phase)) // steps_to_check
    
    data_phase = pd.Series(data_phase, index=indices)
    
    block_averages = data_phase.groupby(data_phase.index).mean()
    
    # calculate the differences between the blocks
    deltas = (block_averages - block_averages.shift(1)).abs()
    score = deltas.mean()
    scores[phase] = score

    # The score will oscillate naturally, 
    if score > max_score * 1.01:
        max_score = score
        max_score_phase = phase

# TODO do this in numpy, dict is lame
print("----------")
print("max_score:", max_score)
print("max_score_phase:", max_score_phase)
    

In [ ]:
px.line(pd.DataFrame(scores.values()))

In [ ]:
# Apply the optimal phase shift
data = data[max_score_phase:]

In [ ]:
# Downsample the data back to the convenient bucketing (1 bucket per 0.5us)
indices = np.arange(len(data)) // (samples_per_us / 2)
data_downsampled = pd.DataFrame(data).groupby(indices).mean()

In [ ]:
fig = px.bar(data_downsampled)
fig.update_traces(marker_line_width = 0,
                  selector=dict(type="bar"))

# Interpreting aircraft signals

In [ ]:
1 / sample_rate

In [ ]:
ADSB_SLOT_LENGTH / 2

In [ ]:
# Assert that we are sampling at the same rate as the signal comms (so we can plot easily later
assert (1 / sample_rate) == ((ADSB_SLOT_LENGTH / 2))

In [ ]:
# Start up the SDR connection
try:
    sdr.close()
except:
    pass

sdr = s_acq.get_sdr(center_freq=center_freq, sample_rate=sample_rate)

s = s_acq.acquire_signal(sdr, n_reads=n_reads, n_samples=n_samples)


In [ ]:
s_df = pd.DataFrame(np.abs(s)).T

# Plot all the reads, see where there were peaks

In [ ]:
px.line(s_df.rolling(16).max()[::64])

### Plot the samples of the best candidate read
Given that this capture had the highest peak magnitude within it, it likely included an ASDB pulse

In [ ]:
# Grab the column with the highest individual peak
signal_col = s_df[s_df.max().idxmax()]

In [ ]:
fig = px.bar(signal_col)

fig.update_traces(marker_line_width = 0,
                  selector=dict(type="bar"))

# fig.update_layout(bargap=0,
#                   bargroupgap = 0,
#                  )


In [ ]:
low_cut = 0.15
signal_col.loc[signal_col < low_cut] = 0.1


In [ ]:
# mode S preamble, 8us
preamble = [1,0,1,0,0,0,0,1,0,1,0,0,0,0,0,0]
assert(len(preamble)==16)
    

In [ ]:
fig = px.bar(signal_col)

fig.update_traces(marker_line_width = 0,
                  selector=dict(type="bar"))

# fig.update_layout(bargap=0,
#                   bargroupgap = 0,
#                  )


# Plot the on/off signalling

- If the peak is one colour, then the signal was in the first half of a slot
- If the peak is the other,  then the signal was in the second half of a slot

On or off depends on the start point, can't be predicted assessed ahead of time

In [ ]:
evens_indexer = signal_col.index % 2 == 0

In [ ]:
# Split the first half/second half into separate columns so they can be plotted differently
even = signal_col.reindex(signal_col.index[evens_indexer])
odd  = signal_col.reindex(signal_col.index[~evens_indexer])
even.name="even"
odd.name="odd"

# Plot the strongest signal to highlight 1s and 0s
- Within each 1us (microsecond, millionth of a second), there are two halves to the 'frame'
- If there is a signal peak within the first half, this is a 1
- If there is a signal peak within the second half, this is a 0

In [ ]:
# fig = px.bar(pd.concat([even, odd], axis=1))

# fig.update_traces(marker_line_width = 0,
#                   selector=dict(type="bar"))

In [ ]:
low_cut = 0.15